In [ ]:
"""
=============================================================================
PROJETO: Classificador de Phishing e Engenharia Social (Fase 3 - Definitiva)
=============================================================================

* INTENÇÃO DA MUDANÇA (HARD NEGATIVES E DATA-CENTRIC AI):
A intenção deste notebook é curar o "Viés de Gatilho" desenvolvido na
Fase 2. Para ensinar a IA que termos financeiros nem sempre indicam um crime,
injetamos um grande volume de mensagens do dia a dia contendo palavras-chave
que antes assustavam o modelo. Isso restabelece o equilíbrio semântico e
foca o BERT na intenção contextual da comunicação.

* DADOS UTILIZADOS:
- Dataset da fase anterior ('dataset_base.csv')[cite: 2].
- Injeção de 996 mensagens de "Hard Negatives" geradas por IA (com
  contexto inofensivo abordando divisões de contas no Pix, encomendas
  reais, pagamento de boletos de condomínio e parabenizações)[cite: 2].
- Dataset consolidado salvo como 'dataset_phishing_v4.csv', gerando o
  treinamento final e mais robusto com 13.046 mensagens balanceadas[cite: 2].

* RESULTADOS PRELIMINARES (ESTADO DA ARTE):
O modelo atingiu o ponto ideal de equilíbrio. No teste com dados complexos
e repletos de gírias, a Acurácia Global subiu para 84%[cite: 2]. Mais importante
ainda, ele obteve um F1-Score de 85,71%[cite: 2], conseguindo diferenciar com
excelente margem um golpe de um pedido real para rachar uma pizza no Pix.
A IA se provou capaz de realizar análises puramente semânticas sem atalhos.

* POSSÍVEIS MELHORIAS:
- Otimização de Hiperparâmetros (Learning Rate, Dropout) para lapidação final.
- Quantização e exportação do modelo para formatos leves (como ONNX ou TFLite)
  para implementação direta em dispositivos móveis (Edge Computing).
- Implementação de auditoria (Shadow Mode) e coleta de feedback orgânico de
  usuários ("Human-in-the-loop") para treinamento de versões futuras.
=============================================================================
"""

In [ ]:
# 1. Instalar a NOVA biblioteca oficial do Google
!pip install -q google-genai pandas
!pip install -q datasets deep-translator pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
# 1. Instalar as bibliotecas necessárias
import json
import time
from google import genai
import pandas as pd
from datasets import load_dataset
# from deep_translator import GoogleTranslator
from tqdm import tqdm
import pandas as pd
from datasets import load_dataset, Dataset
# from deep_translator import GoogleTranslator
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import torch
#!git clone -b desenv https://github.com/jmmfilho/sec-fraudimabr.git

In [ ]:
# 1. Instalar a biblioteca oficial do Google (caso não esteja no ambiente)

# 2. Configure sua chave de API
API_KEY = "AQ.Ab8RN6IOrLAEv8E5jWQA61-9SkVhhgIqpeprek4TvSWn0YwxEw"
client = genai.Client(api_key=API_KEY)

# Use o modelo ativo (ajuste para gemini-2.5-flash ou gemini-3.5-flash dependendo da sua conta)
MODEL_ID = 'gemini-3.5-flash-lite'

# 3. Definição da Meta (Aproximadamente 1.000 mensagens totais para atingir a proporção 80/20)
META_POR_CATEGORIA = 100
MENSAGENS_POR_LOTE = 20

# Categorias desenhadas ESPECIFICAMENTE para desarmar os gatilhos da IA
categorias_hard_negatives = [
    "Rotina Financeira Saudável: Rachar a conta da pizza, pagar consertos reais, mandar mesada. OBRIGATÓRIO usar as palavras: 'pix', 'pagar', 'transferir'.",
    "Contas de Casa: Lembretes entre familiares ou amigos para pagar contas normais do mês. OBRIGATÓRIO usar as palavras: 'boleto', 'fatura', 'vencimento'.",
    "Entregas Reais: Conversas normais pedindo para alguém receber compras na portaria ou comentando de compras legítimas. OBRIGATÓRIO usar as palavras: 'pacote', 'magalu', 'correios', 'entrega'.",
    "Comemorações: Mensagens afetivas e de amizade desejando felicidades por aniversário ou promoção no trabalho. OBRIGATÓRIO usar as palavras: 'parabéns', 'sucesso', 'festa'.",
    "Problemas Cotidianos: Reclamar que esqueceu a senha do Wi-Fi, ou que a própria conta do Insta travou (conversa entre amigos, não suporte). OBRIGATÓRIO usar as palavras: 'senha', 'conta', 'bloqueou'."
]

dados_hard_negatives = []

def criar_prompt_inocente(categoria, quantidade):
    return f"""
    Aja como um usuário comum de WhatsApp no Brasil enviando mensagens para amigos e familiares.
    Gere exatamente {quantidade} mensagens curtas (10 a 30 palavras).

    CENÁRIO OBRIGATÓRIO:
    {categoria}

    REGRAS CRÍTICAS DE SEGURANÇA:
    1. NENHUMA MENSAGEM PODE SER GOLPE. Todas devem ser 100% inocentes e do dia a dia. Não faça ameaças ou cobranças falsas.
    2. Linguagem: Use muito internetês (vc, tb, blz, pdc, pfv, mano, glr), falta de acentuação e pontuação informal.
    3. As mensagens não devem conter links de forma alguma.

    FORMATO DE SAÍDA:
    Retorne APENAS um array JSON válido contendo as strings das mensagens, sem marcações markdown ou títulos. Exemplo: ["msg 1", "msg 2"]
    """

print("Iniciando a geração de Hard Negatives (Mensagens Inofensivas com Gatilhos)...\n")

for categoria in categorias_hard_negatives:
    nome_categoria = categoria.split(":")[0]
    print(f"Gerando mensagens para: {nome_categoria}...")

    mensagens_geradas = 0

    while mensagens_geradas < META_POR_CATEGORIA:
        try:
            prompt = criar_prompt_inocente(categoria, MENSAGENS_POR_LOTE)

            resposta = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt
            )

            # Limpeza do texto para garantir que o JSON será lido corretamente
            texto_limpo = resposta.text.replace("```json", "").replace("```", "").strip()
            lote = json.loads(texto_limpo)

            for msg in lote:
                # O rótulo AQUI É 0 (NORMAL), pois estamos curando os falsos positivos!
                dados_hard_negatives.append({"texto": msg, "label": 0})
                mensagens_geradas += 1

            print(f"  -> +{len(lote)} mensagens ({mensagens_geradas}/{META_POR_CATEGORIA})")
            time.sleep(4) # Pausa para respeitar a cota gratuita da API

        except Exception as e:
            print(f"  -> Erro (Rate Limit ou formatação). Aguardando... Erro: {e}")
            time.sleep(6)

# 4. Salvar os resultados curativos
df_hn = pd.DataFrame(dados_hard_negatives)
df_hn = df_hn.drop_duplicates(subset=['texto'])

nome_arquivo = "hard_negatives.csv"
df_hn.to_csv(nome_arquivo, index=False)

print(f"\n✅ Fábrica concluída! Arquivo curativo salvo como '{nome_arquivo}'.")
print(f"Total de Hard Negatives únicos gerados: {len(df_hn)}")

Iniciando a geração de Hard Negatives (Mensagens Inofensivas com Gatilhos)...

Gerando mensagens para: Rotina Financeira Saudável...
  -> +20 mensagens (20/100)
  -> +20 mensagens (40/100)
  -> +20 mensagens (60/100)
  -> +20 mensagens (80/100)
  -> +20 mensagens (100/100)
Gerando mensagens para: Contas de Casa...
  -> +20 mensagens (20/100)
  -> +20 mensagens (40/100)
  -> +20 mensagens (60/100)
  -> +20 mensagens (80/100)
  -> +20 mensagens (100/100)
Gerando mensagens para: Entregas Reais...
  -> +20 mensagens (20/100)
  -> +20 mensagens (40/100)
  -> +20 mensagens (60/100)
  -> +20 mensagens (80/100)
  -> +20 mensagens (100/100)
Gerando mensagens para: Comemorações...
  -> +20 mensagens (20/100)
  -> +20 mensagens (40/100)
  -> +20 mensagens (60/100)
  -> +20 mensagens (80/100)
  -> +20 mensagens (100/100)
Gerando mensagens para: Problemas Cotidianos...
  -> +20 mensagens (20/100)
  -> +20 mensagens (40/100)
  -> +20 mensagens (60/100)
  -> +20 mensagens (80/100)
  -> +20 mensagens 

In [ ]:


# 1. Carregar o seu dataset atual exato
arquivo_base = "dataset_base.csv"
print(f"Carregando o dataset base: {arquivo_base}...")

try:
    df_base = pd.read_csv(arquivo_base)
    # Garantir que não há valores nulos
    df_base = df_base.dropna(subset=['texto', 'label'])
    print(f"-> Total de mensagens na base atual: {len(df_base)}")
except Exception as e:
    print(f"Erro ao carregar {arquivo_base}: {e}")

# 2. Carregar o arquivo com os Hard Negatives gerados
arquivo_hard_negatives = "hard_negatives.csv"
print(f"\nCarregando os Hard Negatives de {arquivo_hard_negatives}...")

try:
    df_hn = pd.read_csv(arquivo_hard_negatives)
    # Garantir que todas as mensagens novas têm label 0 (Normal)
    df_hn['label'] = 0
    df_hn = df_hn.dropna(subset=['texto'])
    print(f"-> Total de Hard Negatives a serem injetados: {len(df_hn)}")
except Exception as e:
    print(f"Erro ao carregar {arquivo_hard_negatives}. Crie o arquivo primeiro. Erro: {e}")
    # Cria dataframe vazio para não quebrar o código
    df_hn = pd.DataFrame(columns=['texto', 'label'])

# 3. Mesclar (Concatenar) os dois Datasets
print("\nMesclando as bases...")
df_final = pd.concat([df_base, df_hn], ignore_index=True)

# 4. Embaralhar os dados fortemente (Crucial para o treinamento do modelo)
df_final = shuffle(df_final, random_state=42).reset_index(drop=True)

# 5. Salvar a Versão Final (V4)
nome_arquivo_final = "dataset_phishing_v4.csv"
df_final.to_csv(nome_arquivo_final, index=False)

print("\n=======================================================")
print(f"✅ Sucesso! Novo dataset salvo como: '{nome_arquivo_final}'")
print(f"Total de mensagens prontas para treino: {len(df_final)}")
print("\nNova Distribuição de Classes:")
print(df_final['label'].value_counts())
print("=======================================================")

Carregando o dataset base: dataset_base.csv...
-> Total de mensagens na base atual: 17061

Carregando os Hard Negatives de hard_negatives.csv...
-> Total de Hard Negatives a serem injetados: 498

Mesclando as bases...

✅ Sucesso! Novo dataset salvo como: 'dataset_phishing_v4.csv'
Total de mensagens prontas para treino: 17559

Nova Distribuição de Classes:
label
1    11534
0     6025
Name: count, dtype: int64


In [ ]:


print("Iniciando a preparação dos dados...")

# ==========================================
# PASSO 1: PREPARAÇÃO E BALANCEAMENTO
# ==========================================

# 1. Carregar o novo Super Dataset
df = pd.read_csv("dataset_phishing_v4.csv")
df = df.dropna(subset=['texto'])

# Força toda a coluna 'texto' a ser do tipo string (texto)
df['texto'] = df['texto'].astype(str)
# 2. Balanceamento Estrito (Undersampling)
# Descobre qual é a classe com menos exemplos e iguala a outra para ter o mesmo tamanho
tamanho_menor_classe = df['label'].value_counts().min()

df_normal = df[df['label'] == 0].sample(tamanho_menor_classe, random_state=42)
df_golpe = df[df['label'] == 1].sample(tamanho_menor_classe, random_state=42)

# Junta tudo e embaralha
df_balanceado = pd.concat([df_normal, df_golpe]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset balanceado com sucesso! Total: {len(df_balanceado)} mensagens.")
print(df_balanceado['label'].value_counts())

# 3. Divisão de Treino (80%) e Validação (20%)
df_train, df_val = train_test_split(df_balanceado, test_size=0.3, random_state=42)

# As colunas já estão como 'texto' e 'label', vamos renomear para o padrão Hugging Face
df_train.columns = ['text', 'labels']
df_val.columns = ['text', 'labels']

# Converter Pandas para Dataset da Hugging Face
train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
val_dataset = Dataset.from_pandas(df_val, preserve_index=False)

# ==========================================
# PASSO 2: TREINAMENTO (API MODEL)
# ==========================================

# 1. Inicializar Tokenizer e Modelo
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Função de Tokenização
def tokenizar_funcao(exemplos):
    return tokenizer(exemplos["text"], padding=False, truncation=True, max_length=128)

print("Tokenizando os textos...")
tokenized_train = train_dataset.map(tokenizar_funcao, batched=True)
tokenized_val = val_dataset.map(tokenizar_funcao, batched=True)

# 3. Configurar Data Collator (Otimiza a memória da GPU preenchendo os textos dinamicamente)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 4. Configurar Hiperparâmetros
training_args = TrainingArguments(
    output_dir="./resultados_bert_phishing",
    eval_strategy="epoch",             # Avalia no fim de cada época
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4 ,               # 3 épocas é o padrão ouro para BERT
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# 5. Inicializar o Treinador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,        # Parâmetro atualizado (substitui o antigo 'tokenizer')
    data_collator=data_collator
)

# 6. INICIAR O TREINAMENTO
print("Iniciando o treinamento na GPU...")
trainer.train()

# 7. Salvar o modelo final
model.save_pretrained("./meu_modelo_api_phishing")
tokenizer.save_pretrained("./meu_modelo_api_phishing")
print("✅ Modelo salvo com sucesso na pasta './meu_modelo_api_phishing'!")

Iniciando a preparação dos dados...
Dataset balanceado com sucesso! Total: 12050 mensagens.
label
1    6025
0    6025
Name: count, dtype: int64


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (13 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modelo salvo com sucesso na pasta './meu_modelo_api_phishing'!


In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
# Carregar o pipeline de classificação com o modelo que você acabou de treinar
classificador = pipeline(
    "text-classification",
    model="./meu_modelo_api_phishing",
    tokenizer="./meu_modelo_api_phishing",
    device=0 if torch.cuda.is_available() else -1 # Usa GPU se disponível
)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
print("Carregando o modelo salvo...")
# 1. Carregar o modelo treinado diretamente da pasta onde foi salvo
classificador = pipeline(
    "text-classification",
    model="./meu_modelo_api_phishing",
    tokenizer="./meu_modelo_api_phishing",
    device=0 if torch.cuda.is_available() else -1, # Usa GPU para acelerar a validação
    truncation=True,    # Corta textos que forem maiores que o limite
    max_length=128
)

# 2. Extrair os textos e os rótulos (labels) do dataset de validação (df_val)
# Nota: Assumimos que a variável df_val ainda está na memória do Colab.
textos_val = df_val['text'].tolist()
y_true_val = df_val['labels'].tolist()

print(f"Realizando predições em {len(textos_val)} mensagens de validação. Isso pode levar alguns segundos...")
# 3. Fazer as predições em lote
resultados_val = classificador(textos_val)

# 4. Extrair os valores preditos (0 ou 1) da resposta do modelo
y_pred_val = []
for resultado in resultados_val:
    # O modelo retorna 'LABEL_0' ou 'LABEL_1'. Extraímos apenas o número.
    label_previsto = int(resultado['label'].split('_')[-1])
    y_pred_val.append(label_previsto)

# 5. Calcular as métricas
acuracia_val = accuracy_score(y_true_val, y_pred_val)
f1_val = f1_score(y_true_val, y_pred_val, average='binary')

# 6. Exibir o Relatório Completo
print("\n" + "="*50)
print(" RESULTADOS REAIS NO DATASET DE VALIDAÇÃO (20%)")
print("="*50)
print(f"Acurácia Global: {acuracia_val:.4f} ({acuracia_val * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1_val:.4f} ({f1_val * 100:.2f}%)")
print("\nRelatório Detalhado:")
print(classification_report(y_true_val, y_pred_val, target_names=["Normal (0)", "Phishing (1)"]))

Carregando o modelo salvo...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Realizando predições em 3615 mensagens de validação. Isso pode levar alguns segundos...

 RESULTADOS REAIS NO DATASET DE VALIDAÇÃO (20%)
Acurácia Global: 0.9956 (99.56%)
F1-Score (Fraudes): 0.9955 (99.55%)

Relatório Detalhado:
              precision    recall  f1-score   support

  Normal (0)       1.00      0.99      1.00      1845
Phishing (1)       0.99      1.00      1.00      1770

    accuracy                           1.00      3615
   macro avg       1.00      1.00      1.00      3615
weighted avg       1.00      1.00      1.00      3615



In [ ]:
# 1. Modificando as mensagens para incluir o rótulo real (Ground Truth)
# Formato: ("Texto da mensagem", rótulo_real)
# Assumindo que 0 = Normal e 1 = Phishing (verifique a ordem do seu CSV)
mensagens_teste = [
    # --- MENSAGENS NORMAIS (Label 0) ---
    ("Oi, não esquece de passar na padaria voltando do trabalho.", 0),
    ("A reunião foi transferida para a sala 3, avisem os demais.", 0),
    ("Você viu o jogo ontem? Que golaço no final!", 0),
    ("Seu pedido Magalu 4920 foi entregue ao destinatário.", 0),
    ("Feliz Natal, família! Que a noite de vocês seja iluminada.", 0),
    ("Preciso que você me envie o PDF daquela apresentação até meio-dia.", 0),
    ("Estou no trânsito, devo atrasar uns 10 minutinhos.", 0),
    ("Bom dia, o valor do conserto ficou em R$ 150. Pode fazer o Pix para a chave celular.", 0),
    ("Lembrete: Consulta com o Dr. Marcos amanhã às 15:00.", 0),
    ("Bora marcar um futebol no fim de semana?", 0),
    ("Amor, a ração do cachorro acabou, compra mais depois.", 0),
    ("A sua fatura da Vivo com vencimento em 10/08 está disponível no app.", 0),
    ("Mãe, cheguei bem no hotel. Mais tarde te ligo.", 0),
    ("Alguém tem o contato de um bom encanador pra me passar?", 0),
    ("A professora de inglês mandou o link da aula no grupo.", 0),
    ("Oi, desculpa a demora pra responder, tava em reunião.", 0),
    ("Você vai almoçar em casa hoje ou no restaurante?", 0),
    ("Parabéns, amigo! Muitas felicidades e sucesso sempre.", 0),
    ("Uber: Seu motorista Carlos está chegando em 2 minutos.", 0),
    ("O boleto do condomínio já foi enviado pro seu email.", 0),
    ("Vou viajar no feriado, depois a gente se vê.", 0),
    ("Tem pão fresco saindo agora, vai querer que eu leve?", 0),
    ("A vacinação da gripe começou no posto aqui da esquina.", 0),
    ("Não consegui falar com você, me liga quando puder.", 0),
    ("Confirmando o nosso jantar de amanhã às 20h. Até lá!", 0),

    # --- MENSAGENS DE PHISHING / FRAUDE (Label 1) ---
    ("BANCO CENTRAL: Há um Pix de R$ 850 retido em seu nome. Acesse para liberar: ", 1),
    ("C6 Bank: Seu cartão foi bloqueado por segurança. Valide sua identidade imediatamente: ", 1),
    ("Mãe, meu celular quebrou, salva esse meu número novo provisório.", 1),
    ("Olá, somos do RH da Amazon. Temos vaga de meio período para ganhar de 200 a 500 por dia. Clique: ", 1),
    ("Detran informa: Suspensão de CNH agendada. Pague o débito para evitar o cancelamento: ", 1),
    ("Aviso gov.br: Sua conta será desativada em 2h. Atualize seus dados cadastrais em ", 1),
    ("CORREIOS: Seu pedido foi retido na alfândega. Taxa de R$ 22,50 pendente. Pague aqui: ", 1),
    ("Netflix: O pagamento do mês falhou. Atualize seu cartão de crédito para não perder o acesso: ", 1),
    ("Promoção O Boticário! Responda o questionário e ganhe um kit grátis de Dia das Mães: ", 1),
    ("Itaú: Identificamos um acesso no Rio de Janeiro. Caso não tenha sido você, conteste em [URL]", 1),
    ("Você tem 20.000 pontos Livelo expirando hoje! Troque por milhas ou produtos em ", 1),
    ("Serasa Consumidor: Pague sua dívida com 95% de desconto no Feirão Limpa Nome. Pix oficial: ", 1),
    ("O Governo aprovou um Auxílio Gás de R$ 120. Consulte se seu CPF tem direito: [URL]", 1),
    ("Oi, encontrei essas fotos suas da festa de ontem rolando na internet, dá uma olhada [URL] ", 1),
    ("Shopee: Parabéns, você foi selecionado para avaliar produtos. Renda diária de R$ 300. Chame no WhatsApp: ", 1),
    ("Nubank: Sua compra no valor de R$ 2.500 no MercadoPago foi aprovada. Para cancelar, clique: ", 1),
    ("Aviso Receita Federal: Sua declaração de imposto de renda caiu na malha fina. Veja a divergência: ", 1),
    ("WhatsApp: Insira o código de 6 dígitos que você recebeu por SMS para não perder sua conta.", 1),
    ("Caixa Tem: Seu saque extraordinário foi liberado. Acesse a conta para transferir o valor: ", 1),
    ("Santander: Seu aplicativo de segurança está desatualizado. Evite multas atualizando no link: ", 1),
    ("Oportunidade Única! Invista R$ 50 no nosso robô de Pix e receba R$ 500 em 10 minutos. Garanta sua vaga: ", 1),
    ("Claro: Sua fatura em aberto pode ser paga com 50% de desconto hoje no Pix. Gere a chave em: ", 1),
    ("Atenção: Ação judicial protocolada em seu CPF. Visualize o processo e mandado de intimação aqui: ", 1),
    ("Empréstimo liberado de R$ 10.000 para negativados sem consulta ao SPC! Assine o contrato online: ", 1),
    ("Mercado Livre: Seu acesso está suspenso temporariamente por atividade suspeita. Confirme sua titularidade", 1)
]

# 2. Separar os textos e os rótulos verdadeiros para processamento
textos = [item[0] for item in mensagens_teste]
y_true = [item[1] for item in mensagens_teste]

# 3. Fazer as predições usando o pipeline da Hugging Face
resultados = classificador(textos)

# 4. Extrair as predições do modelo (y_pred)
y_pred = []
print("--- DETALHAMENTO DAS MENSAGENS ---\n")
for texto, real, resultado in zip(textos, y_true, resultados):
    # O modelo retorna 'LABEL_0' ou 'LABEL_1'. Pegamos apenas o número final e convertemos para inteiro.
    label_previsto = int(resultado['label'].split('_')[-1])
    y_pred.append(label_previsto)

    print(f"Mensagem: {texto}")
    print(f"Real: {real} | Previsto: {label_previsto} | Certeza: {resultado['score']:.4f}\n")

# 5. Calcular e exibir as métricas
acuracia = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='binary') # 'binary' é ideal para problemas de duas classes

print("="*40)
print(" RESULTADOS GERAIS DA AVALIAÇÃO")
print("="*40)
print(f"Acurácia Global: {acuracia:.4f} ({acuracia * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1:.4f} ({f1 * 100:.2f}%)")
print("\nRelatório Detalhado:")
print(classification_report(y_true, y_pred, target_names=["Normal (0)", "Phishing (1)"]))

--- DETALHAMENTO DAS MENSAGENS ---

Mensagem: Oi, não esquece de passar na padaria voltando do trabalho.
Real: 0 | Previsto: 0 | Certeza: 0.9989

Mensagem: A reunião foi transferida para a sala 3, avisem os demais.
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: Você viu o jogo ontem? Que golaço no final!
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: Seu pedido Magalu 4920 foi entregue ao destinatário.
Real: 0 | Previsto: 0 | Certeza: 0.9797

Mensagem: Feliz Natal, família! Que a noite de vocês seja iluminada.
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: Preciso que você me envie o PDF daquela apresentação até meio-dia.
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: Estou no trânsito, devo atrasar uns 10 minutinhos.
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: Bom dia, o valor do conserto ficou em R$ 150. Pode fazer o Pix para a chave celular.
Real: 0 | Previsto: 0 | Certeza: 0.9558

Mensagem: Lembrete: Consulta com o Dr. Marcos amanhã às 15:00.
Real: 0 | Previs

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
mensagens_teste_2 = [
    # --- MENSAGENS NORMAIS (Label 0) ---
    ("oi, n esquece de passar na padaria voltando do trampo", 0),
    ("a reunião mudou pra sala 3, avisa a glr pfv", 0),
    ("vc viu o jogo ontem? q golaço no final slc!", 0),
    ("seu pacote do magalu chegou aqui na portaria, dps pega la", 0),
    ("feliz natal fml!! q a noite de vcs seja top", 0),
    ("preciso q vc me mande o pdf daquela apresentaçao ate umas 12h", 0),
    ("to no transito, vou atrasar uns 10 min malz", 0),
    ("bom dia! o conserto ficou 150 conto. pd fzr o pix no meu cel msm", 0),
    ("oie, passando p/ lembrar da consulta cm o dr marcos amanhã as 15h blz?", 0),
    ("bora marcar um fut esse fds?", 0),
    ("amor, a raçao do dog acabou, compra + dps", 0),
    ("a fatura da vivo ja ta no app, n esquece de pagar", 0),
    ("mae cheguei dboa no hotel. mais tarde te ligo bjs", 0),
    ("alguem tem ctt de um encanador bom pra me passar?", 0),
    ("a prof de ingles mandou o link da aula no gp", 0),
    ("oi, dsclp a demora pra responder, tava em reuniao", 0),
    ("vc vai almoçar em ksa hj ou vai comer fora?", 0),
    ("parabens mano!! mtas felicidades e sucesso smp tmj", 0),
    ("o uber ta chegando, 2 minutinhos", 0),
    ("o boleto do cond ja ta no seu email", 0),
    ("vou viajar no feriado, dps a gnt se ve", 0),
    ("tem pão fresco saindo agr, qr q eu leve?", 0),
    ("a vacina da gripe começou no posto aq da esquina", 0),
    ("n consegui falar c/ vc, me liga qnd der", 0),
    ("confirmando nosso jantar de amanhã as 20h. ate!", 0),

    # --- MENSAGENS DE PHISHING / FRAUDE (Label 1) ---
    ("Bacen: tem um pix de 850 conto retido no seu cpf. acessa p/ liberar: ", 1),
    ("C6 Bank: cartao bloqueado por seguranca. valida ai rapidao senao cancela: ", 1),
    ("Oii mae, meu cel quebrou :( salva meu num novo provisorio ai", 1),
    ("Opa, sou do RH da Amazon. Vaga meio periodo tirando de 200 a 500/dia. Clica ai: ", 1),
    ("Detran: suspensao de CNH agendada. Paga o debito p/ evitar BO no doc: ", 1),
    ("gov.br: sua conta vai ser desativada hj! atualiza seus dados aqui ", 1),
    ("Correios: seu pedido ficou retido na alfandega. Tem taxa de 22,50 pendente no pix: ", 1),
    ("Netflix: o pgto falhou :( atualiza o cartao pra n perder o acesso: ", 1),
    ("Promoção O Boticário bombando! responde a pesquisa e ganha 1 kit de dia das mães free: ", 1),
    ("Itau: acesso em outro estado detectado. Nao foi vc? Contesta aq no link [URL]", 1),
    ("vc tem 20k de pontos Livelo expirando HOJE!! troca ai por milha no link ", 1),
    ("Serasa: limpe seu nome c/ 95% off no feirao! paga no pix oficial aq: ", 1),
    ("Governo liberou Auxilio Gas de 120$!! ve se seu cpf tem direito [URL]", 1),
    ("nossa, vc viu as fotos suas da festa q vazaram?? q loucura, olha ai [URL] ", 1),
    ("Oie, vc foi selecionado p/ avaliar produtos na Shopee e tirar +300/dia. Chama no whats: ", 1),
    ("Nubank: compra de 2.500 aprovada no mercadopago. nao reconhece? clica aq pra cancelar: ", 1),
    ("Receita Federal: seu IRPF caiu na malha fina. acesse pra ver oq deu errado: ", 1),
    ("Suporte WhatsApp: digite o cod q vc recebeu por sms pra n perder o whats.", 1),
    ("Caixa Tem: saque extraordinario na conta!! clica pra transferir o dindin: ", 1),
    ("Santander: app desatualizado. atualiza no link p/ n tomar multa no cpf: ", 1),
    ("quer fzr renda extra?? investe 50tao no robo do pix e tira 500 na hora. clica: ", 1),
    ("Claro: fatura em aberto com 50% de desc hj no pix. gera a chave ai: ", 1),
    ("Atenção! açao judicial no seu CPF. ve o processo e o mandado aqui: ", 1),
    ("emprestimo de 10k aprovado ate pra negativado slc!! s/ consulta no spc. assina ai: ", 1),
    ("Mercado Livre: conta suspensa p/ ativ suspeita. confirma q é vc aqui ", 1),
        ("Oi, não esquece de passar na padaria voltando do trabalho.", 0),
    ("A reunião foi transferida para a sala 3, avisem os demais.", 0),
    ("Você viu o jogo ontem? Que golaço no final!", 0),
    ("Seu pedido Magalu 4920 foi entregue ao destinatário.", 0),
    ("Feliz Natal, família! Que a noite de vocês seja iluminada.", 0),
    ("Preciso que você me envie o PDF daquela apresentação até meio-dia.", 0),
    ("Estou no trânsito, devo atrasar uns 10 minutinhos.", 0),
    ("Bom dia, o valor do conserto ficou em R$ 150. Pode fazer o Pix para a chave celular.", 0),
    ("Lembrete: Consulta com o Dr. Marcos amanhã às 15:00.", 0),
    ("Bora marcar um futebol no fim de semana?", 0),
    ("Amor, a ração do cachorro acabou, compra mais depois.", 0),
    ("A sua fatura da Vivo com vencimento em 10/08 está disponível no app.", 0),
    ("Mãe, cheguei bem no hotel. Mais tarde te ligo.", 0),
    ("Alguém tem o contato de um bom encanador pra me passar?", 0),
    ("A professora de inglês mandou o link da aula no grupo.", 0),
    ("Oi, desculpa a demora pra responder, tava em reunião.", 0),
    ("Você vai almoçar em casa hoje ou no restaurante?", 0),
    ("Parabéns, amigo! Muitas felicidades e sucesso sempre.", 0),
    ("Uber: Seu motorista Carlos está chegando em 2 minutos.", 0),
    ("O boleto do condomínio já foi enviado pro seu email.", 0),
    ("Vou viajar no feriado, depois a gente se vê.", 0),
    ("Tem pão fresco saindo agora, vai querer que eu leve?", 0),
    ("A vacinação da gripe começou no posto aqui da esquina.", 0),
    ("Não consegui falar com você, me liga quando puder.", 0),
    ("Confirmando o nosso jantar de amanhã às 20h. Até lá!", 0),

    # --- MENSAGENS DE PHISHING / FRAUDE (Label 1) ---
    ("BANCO CENTRAL: Há um Pix de R$ 850 retido em seu nome. Acesse para liberar: ", 1),
    ("C6 Bank: Seu cartão foi bloqueado por segurança. Valide sua identidade imediatamente: ", 1),
    ("Mãe, meu celular quebrou, salva esse meu número novo provisório.", 1),
    ("Olá, somos do RH da Amazon. Temos vaga de meio período para ganhar de 200 a 500 por dia. Clique: ", 1),
    ("Detran informa: Suspensão de CNH agendada. Pague o débito para evitar o cancelamento: ", 1),
    ("Aviso gov.br: Sua conta será desativada em 2h. Atualize seus dados cadastrais em ", 1),
    ("CORREIOS: Seu pedido foi retido na alfândega. Taxa de R$ 22,50 pendente. Pague aqui: ", 1),
    ("Netflix: O pagamento do mês falhou. Atualize seu cartão de crédito para não perder o acesso: ", 1),
    ("Promoção O Boticário! Responda o questionário e ganhe um kit grátis de Dia das Mães: ", 1),
    ("Itaú: Identificamos um acesso no Rio de Janeiro. Caso não tenha sido você, conteste em [URL]", 1),
    ("Você tem 20.000 pontos Livelo expirando hoje! Troque por milhas ou produtos em ", 1),
    ("Serasa Consumidor: Pague sua dívida com 95% de desconto no Feirão Limpa Nome. Pix oficial: ", 1),
    ("O Governo aprovou um Auxílio Gás de R$ 120. Consulte se seu CPF tem direito: [URL]", 1),
    ("Oi, encontrei essas fotos suas da festa de ontem rolando na internet, dá uma olhada [URL] ", 1),
    ("Shopee: Parabéns, você foi selecionado para avaliar produtos. Renda diária de R$ 300. Chame no WhatsApp: ", 1),
    ("Nubank: Sua compra no valor de R$ 2.500 no MercadoPago foi aprovada. Para cancelar, clique: ", 1),
    ("Aviso Receita Federal: Sua declaração de imposto de renda caiu na malha fina. Veja a divergência: ", 1),
    ("WhatsApp: Insira o código de 6 dígitos que você recebeu por SMS para não perder sua conta.", 1),
    ("Caixa Tem: Seu saque extraordinário foi liberado. Acesse a conta para transferir o valor: ", 1),
    ("Santander: Seu aplicativo de segurança está desatualizado. Evite multas atualizando no link: ", 1),
    ("Oportunidade Única! Invista R$ 50 no nosso robô de Pix e receba R$ 500 em 10 minutos. Garanta sua vaga: ", 1),
    ("Claro: Sua fatura em aberto pode ser paga com 50% de desconto hoje no Pix. Gere a chave em: ", 1),
    ("Atenção: Ação judicial protocolada em seu CPF. Visualize o processo e mandado de intimação aqui: ", 1),
    ("Empréstimo liberado de R$ 10.000 para negativados sem consulta ao SPC! Assine o contrato online: ", 1),
    ("Mercado Livre: Seu acesso está suspenso temporariamente por atividade suspeita. Confirme sua titularidade", 1)
]

textos = [item[0] for item in mensagens_teste_2]
y_true = [item[1] for item in mensagens_teste_2]

# 3. Fazer as predições usando o pipeline da Hugging Face
resultados = classificador(textos)

# 4. Extrair as predições do modelo (y_pred)
y_pred = []
print("--- DETALHAMENTO DAS MENSAGENS ---\n")
for texto, real, resultado in zip(textos, y_true, resultados):
    # O modelo retorna 'LABEL_0' ou 'LABEL_1'. Pegamos apenas o número final e convertemos para inteiro.
    label_previsto = int(resultado['label'].split('_')[-1])
    y_pred.append(label_previsto)

    print(f"Mensagem: {texto}")
    print(f"Real: {real} | Previsto: {label_previsto} | Certeza: {resultado['score']:.4f}\n")

# 5. Calcular e exibir as métricas
acuracia = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='binary') # 'binary' é ideal para problemas de duas classes

print("="*40)
print(" RESULTADOS GERAIS DA AVALIAÇÃO")
print("="*40)
print(f"Acurácia Global: {acuracia:.4f} ({acuracia * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1:.4f} ({f1 * 100:.2f}%)")
print("\nRelatório Detalhado:")
print(classification_report(y_true, y_pred, target_names=["Normal (0)", "Phishing (1)"]))

matriz = confusion_matrix(y_true, y_pred)

# Imprime a versão em texto (modo raiz)
print("Matriz de Confusão:")
print(f"Verdadeiros Negativos: {matriz[0][0]}  | Falsos Positivos: {matriz[0][1]}")
print(f"Falsos Negativos: {matriz[1][0]}       | Verdadeiros Positivos: {matriz[1][1]}")

--- DETALHAMENTO DAS MENSAGENS ---

Mensagem: oi, n esquece de passar na padaria voltando do trampo
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: a reunião mudou pra sala 3, avisa a glr pfv
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: vc viu o jogo ontem? q golaço no final slc!
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: seu pacote do magalu chegou aqui na portaria, dps pega la
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: feliz natal fml!! q a noite de vcs seja top
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: preciso q vc me mande o pdf daquela apresentaçao ate umas 12h
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: to no transito, vou atrasar uns 10 min malz
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: bom dia! o conserto ficou 150 conto. pd fzr o pix no meu cel msm
Real: 0 | Previsto: 0 | Certeza: 0.9998

Mensagem: oie, passando p/ lembrar da consulta cm o dr marcos amanhã as 15h blz?
Real: 0 | Previsto: 0 | Certeza: 0.9999

Mensagem: bora marc